# MapBiomas land use and land cover of 2023 for Figure 1 (study area)

**FINN_SC · `gee/mapbiomas_lulc2023_figure1.ipynb`** (co-authors' review of 2026-10-19, comment C4)

Figure 1 of the manuscript shows the MapBiomas Collection 10 map of 2019, while the land-cover analysis (section 2.8)
uses 2023. This notebook exports the Santa Catarina window of the 2023 map and writes, with the same names and formats
as the 2019 files, the two inputs of `tools/study_area_map.py`:

* `data/external/mapbiomas_c10_lulc2023_SC_120m.npz` (class id at ~120 m, `inside` mask, affine), by
  `tools/extract_mapbiomas_sc.py` with FACTOR 4, as for 2019;
* `data/external/mapbiomas_c10_lulc2023_SC_class_areas.csv` (class areas inside the State at ~60 m, pixel areas by
  latitude, shares of the classified area excluding class 0).

Then: `FIG1_YEAR=2023 python tools/study_area_map.py` redraws Figure 1 and writes
`results/tables/landcover2023_mesoregion.csv`.

Asset: `projects/mapbiomas-public/assets/brazil/lulc/v1` (the same as the municipal fractions of section 2.8), image of
2023, band `classification`, exported in its native grid (EPSG:4326, ~30 m) over the State bounds ± 0.05°.

In [1]:
# 1. Packages (skip if already installed)
%pip install -q earthengine-api geemap geopandas rasterio pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 23.8 MB/s eta 0:00:00


In [2]:
# 2. Parameters
from pathlib import Path
# repository root = parent of gee/ (local run); otherwise the current folder (e.g. Colab, with the input files uploaded)
REPO_ROOT = Path.cwd().resolve().parent
if not (REPO_ROOT / "config" / "config.yml").is_file():
    REPO_ROOT = None
PROJECT = "ee-jessicajcss"   # <-- Earth Engine Cloud project (change it to your own)
YEAR = 2023
COLLECTION_ID = None             # e.g. 10.1 to pin the collection; None = first image of the year (as in section 2.8)
TIF = f"brazil_coverage_{YEAR}_SC.tif"

In [3]:
# 3. Earth Engine authentication
import ee
try:
    ee.Initialize(project=PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT)
print("Earth Engine initialised, project", PROJECT)

Earth Engine initialised, project ee-jessicajcss


In [4]:
# 4. Municipal polygons of Santa Catarina: IBGE 2024 mesh, the same used for the FINN and MapBiomas Fogo tables.
#    Earth Engine receives the simplified copy gee/SC_municipios_2024_simplified.gpkg (tools/simplify_municipios_for_gee.py:
#    shared borders simplified at 0.0005 degrees, about 55 m; area change per municipality <= 0.07 %), because the full
#    shapefile (about 70 MB as GeoJSON) exceeds the 10 MB limit of an Earth Engine request.
#    In Colab, this cell asks for that one file (folder gee/ of the repository, about 2.7 MB).
import geopandas as gpd, geemap, pandas as pd
GPKG = "SC_municipios_2024_simplified.gpkg"
if REPO_ROOT is not None:
    gpkg_path = REPO_ROOT / "gee" / GPKG
else:
    gpkg_path = Path.cwd() / GPKG
    if not gpkg_path.is_file():
        try:
            from google.colab import files
            print(f"Select the file {GPKG} (folder gee/ of the repository)")
            for name in files.upload():
                if name != GPKG and name.startswith("SC_municipios_2024_simplified"):
                    Path(name).rename(GPKG)          # Colab renames a repeated upload, e.g. "... (1).gpkg"
        except ImportError:
            pass
    assert gpkg_path.is_file(), f"copy {GPKG} (folder gee/ of the repository) to {Path.cwd()}"
mun_s = gpd.read_file(gpkg_path)
mun_s["CD_MUN"] = mun_s["CD_MUN"].astype("int64")
assert len(mun_s) == 295 and mun_s.CD_MUN.is_unique and mun_s.mesorregiao.notna().all(), len(mun_s)
if REPO_ROOT is not None:   # full-resolution mesh for the local steps
    mun = gpd.read_file(REPO_ROOT / "data" / "inputs" / "SC_Municipios_2024.shp", encoding="cp1252").to_crs(4326)
    mun["CD_MUN"] = mun["CD_MUN"].astype("int64")
    mun = mun.merge(mun_s[["CD_MUN", "mesorregiao"]], on="CD_MUN", how="left")
else:
    mun = mun_s
mun_ee = geemap.geopandas_to_ee(mun_s[["CD_MUN", "NM_MUN", "mesorregiao", "geometry"]])
print(len(mun_s), "municipalities;", mun_s.groupby("mesorregiao").size().to_dict())


Select the file SC_municipios_2024_simplified.gpkg (folder gee/ of the repository)


Saving SC_municipios_2024_simplified.gpkg to SC_municipios_2024_simplified.gpkg
295 municipalities; {'Grande Florianópolis': 21, 'Norte Catarinense': 26, 'Oeste Catarinense': 118, 'Serrana': 30, 'Sul Catarinense': 46, 'Vale do Itajaí': 54}


In [5]:
# 5. Export the State window of the 2023 map (native grid) to Google Drive
col = ee.ImageCollection("projects/mapbiomas-public/assets/brazil/lulc/v1").filter(ee.Filter.eq("year", YEAR))
if COLLECTION_ID is not None:
    col = col.filter(ee.Filter.eq("collection_id", COLLECTION_ID))
img = col.first().select("classification").toByte()
print("image:", ee.Dictionary({"collection_id": img.get("collection_id"), "version": img.get("version")}).getInfo())
minx, miny, maxx, maxy = mun.total_bounds
region = ee.Geometry.Rectangle([minx - 0.05, miny - 0.05, maxx + 0.05, maxy + 0.05])
p = img.projection().getInfo()
task = ee.batch.Export.image.toDrive(image=img.clip(region), description=TIF[:-4], fileNamePrefix=TIF[:-4], region=region,
                                     crs=p["crs"], crsTransform=p["transform"], maxPixels=1e10, fileFormat="GeoTIFF")
task.start()
print("export started; when it finishes, download", TIF, "from Google Drive to this folder (or to the repository root)")

image: {'collection_id': 10.1, 'version': 'v1'}
export started; when it finishes, download brazil_coverage_2023_SC.tif from Google Drive to this folder (or to the repository root)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [6]:
# 6. After the download: the ~120 m window (same tool and factor as for 2019). Steps 6 and 7 run inside the
#    repository (they use the full-resolution mesh and tools/extract_mapbiomas_sc.py): in Colab, stop here, download
#    the GeoTIFF from Google Drive and save it in the root folder of wildfire_repo.
if REPO_ROOT is None:
    raise SystemExit(f"Colab run finished: download {TIF} from Google Drive to the root folder of wildfire_repo "
                     "and run cells 6 and 7 there")
import subprocess, sys
assert Path(TIF).is_file(), f"download {TIF} first"
root = REPO_ROOT or Path.cwd()
npz = root / "data" / "external" / f"mapbiomas_c10_lulc{YEAR}_SC_120m.npz"
npz.parent.mkdir(parents=True, exist_ok=True)
tool = (REPO_ROOT / "tools" / "extract_mapbiomas_sc.py") if REPO_ROOT else Path("extract_mapbiomas_sc.py")
subprocess.run([sys.executable, str(tool), TIF, str(npz), "4"], check=True)
print("saved", npz)

SystemExit: Colab run finished: download brazil_coverage_2023_SC.tif from Google Drive to the root folder of wildfire_repo and run cells 6 and 7 there

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# 7. Class areas inside the State at ~60 m (factor 2), pixel areas by latitude on the authalic sphere
import numpy as np, rasterio
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
sc = mun.dissolve().geometry.iloc[0]
with rasterio.open(TIF) as r:
    h, w = r.height // 2, r.width // 2
    a = r.read(1, out_shape=(h, w), resampling=Resampling.nearest)
    t = r.transform * rasterio.Affine.scale(r.width / w, r.height / h)
inside = geometry_mask([sc], out_shape=a.shape, transform=t, invert=True)
# WGS84 authalic latitude and radius
A_, F_ = 6378137.0, 1 / 298.257223563; E2 = F_ * (2 - F_); E_ = np.sqrt(E2)
q = lambda phi: (1 - E2) * (np.sin(phi) / (1 - E2 * np.sin(phi) ** 2) - np.log((1 - E_ * np.sin(phi)) / (1 + E_ * np.sin(phi))) / (2 * E_))
QP = q(np.pi / 2); RA = A_ * np.sqrt(QP / 2)
lat_top = t.f + t.e * np.arange(h); lat_bot = lat_top + t.e
beta = lambda lat: np.arcsin(q(np.radians(lat)) / QP)
row_area = RA ** 2 * np.radians(abs(t.a)) * np.abs(np.sin(beta(lat_top)) - np.sin(beta(lat_bot)))   # m2 per pixel, by row
area = np.repeat(row_area[:, None], w, axis=1)
cls = np.unique(a[inside])
tab = pd.DataFrame({"class_id": cls, "area_km2": [area[inside & (a == c)].sum() / 1e6 for c in cls]})
classified = tab[tab.class_id != 0].area_km2.sum()
tab["pct_classified"] = np.where(tab.class_id != 0, 100 * tab.area_km2 / classified, np.nan)
names = pd.read_csv(root / "data" / "external" / "mapbiomas_c10_lulc2019_SC_class_areas.csv")[["class_id", "class_name"]]
tab = tab.merge(names, on="class_id", how="left")[["class_id", "class_name", "area_km2", "pct_classified"]]
out = root / "data" / "external" / f"mapbiomas_c10_lulc{YEAR}_SC_class_areas.csv"
tab.round({"area_km2": 2, "pct_classified": 3}).to_csv(out, index=False)
print(tab.round(2).to_string(index=False)); print("saved", out)

## Next steps

1. From the repository root: `FIG1_YEAR=2023 python tools/study_area_map.py` (Figure 1 in Portuguese and English and
   `results/tables/landcover2023_mesoregion.csv`).
2. Replace Figure 1 in the manuscript, update the year in its caption and the percentages of section 2.1 with the new
   class-area table and the mesoregion table, and record the SHA-256 of the two new files in `docs/provenance.md`.